# Trabajo Práctico Aprendizaje Automático 2 - Predicción de puntuación y capitalización en texto normalizado

In [2]:
import os
import re
from datasets import load_dataset
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from transformers import BertModel, BertTokenizer
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import unicodedata


from dataclasses import dataclass

In [3]:
# Seteamos la semilla para que los resultados sean reproducibles
SEED = 2026
np.random.seed(SEED)

In [6]:
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p

# ejemplo


In [15]:
tokens_ej = tokenizer.tokenize("hola wawa")
token_ids = tokenizer.convert_tokens_to_ids(tokens_ej)
embedin_vec = model.embeddings.word_embeddings.weight

print(tokens_ej)
print(token_ids)
#print(embedin_vec[token_ids[0]])

['hol', '##a', 'wa', '##wa']
[110516, 10113, 11471, 11037]


# constantes


documento = lista de oraciones ? 

descartar oraciones q tengan punto

no mapeamos signos 

descartar cosas q no tengan . o ? o q no tengan ?balanceados .preguntar


In [ ]:
num_oraciones_por_dataset = 5000  

min_palabra_oracion = 3
max_palabra_oracion = 60
max_oracion_en_parrafo = 20
# cada parrafo lo corto en recortes
min_palabra_instancia = 10
max_palabra_instancia = 50
proba_loca = 0.15 #creo que para algunas no oraciones, no cortarlas

proporcio_test = 0.2

RUTA_DATOS = "datos"  

# Cosas


In [18]:
ABREVIATURAS = {
    "sr.", "sra.", "sres.", "sras.", "srta.", "dr.", "dra.", "ud.", "uds.",
    "art.", "arts.", "núm.", "nº.", "pág.", "págs.", "vol.", "cap.",
    "etc.", "ej.", "p.ej.", "aprox.", "ee.uu.", "a.c.", "d.c.",
}

CARACTERES_PERMITIDOS = set(" ¿?.,'-")


In [19]:

def descartar_caca(texto):#return si descarto
    cacas = ["...","…" ,"http","www", "@" ,]
    """Casos que no vale la pena intentar limpiar."""
    for caca in cacas:
        if caca in texto:
            return True
    if re.search(r"\d[.,]\d", texto):#si hay numeros "1.500", "2,5"
        return True
    if any(palabra.lower() in ABREVIATURAS for palabra in texto.split(" ")):#si alguno es
        return True
    if re.search(r"\b[A-Z ÁÉÍÓÚÑ]\.\s", texto):#? (ej: "J. M. Barroso")
        return True
    
    return False


def normalizar_espacios_puntuacion(texto):
    texto = re.sub(r"\s+([?.,])", r"\1", texto)#saca espaceo antes de los .,?,,
    # Agrega un espacio después de '?', '.' o ',' si quedaron pegados (ej: "tarde,hubo")
    texto = re.sub(r"([?.,])(?=\w)", r"\1 ", texto)##pone espacio despues del ,,.,?
    texto = re.sub(r"¿\s+", "¿", texto)#creemos que saca espacios despues del ?{^-1} = ¿
    texto = re.sub(r"(?<=\S)¿", " ¿", texto)#nose
    texto = re.sub(r"\s+", " ", texto).strip() # Elimina espacios múltiples
    return texto


def es_oracion_valida(oracion):#es na oracion valida??? despues d e normalizar 
    palabras = oracion.split(" ")
    if not (min_palabra_oracion <= len(palabras) and len(palabras) <= max_palabra_oracion):
        return False

    if any(not (c.isalnum() or c in CARACTERES_PERMITIDOS) for c in oracion):#no hay letras raras
        return False

    # Tiene que parecer una oración completa: empieza en mayúscula y termina en . o ?
    primera_letra = palabras[0][0] if len(palabras) > 0 else ""#next((c for c in texto if c.isalpha()), "")
    if not primera_letra.isupper():
        return False
    if not (oracion.endswith(".") or oracion.endswith("?")):
        return False

    return True


def limpiar_oracion(texto):
    """Devuelve la oración limpia, o None si hay que descartarla."""
    texto = texto.strip()
    if descartar_caca(texto):
        return None
    texto = normalizar_espacios_puntuacion(texto)
    if not es_oracion_valida(texto):
        return None
    return texto

In [24]:
ejemplo_oraciones = [
    "Y prefiero mudarme, ¿Entiendes?",
    "- Gracias.",
    "Sería -me parece a mí- lo mínimo, al alba del tercer milenio.",
    " Esta tarde,hubo conciertos en las ciudades ocupadas.",
    "Éstas ofrecen tres garantías: ningún petrolero podrá hacer escala; todas las partes deberán responder.",
    "Vamos a McDonald's con el Sr. Pérez.",
]

for ejemplo in ejemplo_oraciones:
    print(f"-{ejemplo!r}\n-{limpiar_oracion(ejemplo)!r}\n")

-'Y prefiero mudarme, ¿Entiendes?'
-'Y prefiero mudarme, ¿Entiendes?'

-'- Gracias.'
-None

-'Sería -me parece a mí- lo mínimo, al alba del tercer milenio.'
-'Sería -me parece a mí- lo mínimo, al alba del tercer milenio.'

-' Esta tarde,hubo conciertos en las ciudades ocupadas.'
-'Esta tarde, hubo conciertos en las ciudades ocupadas.'

-'Éstas ofrecen tres garantías: ningún petrolero podrá hacer escala; todas las partes deberán responder.'
-None

-"Vamos a McDonald's con el Sr. Pérez."
-None



In [ ]:
def leer_documento(nombre_dataset):
    #leo dos datasets que estan en anda a saber donde 
    stream = load_dataset(nombre_dataset, "en-es", split="train", streaming=True)

    documento = []
    documento_actual = []
    n_oraciones = 0

    for ejemplo in stream:
        oracion = limpiar_oracion(ejemplo["translation"]["es"])

        if oracion is None:
            if documento_actual:
                documento.append(documento_actual)
                documento_actual = []
            continue

        documento_actual.append(oracion)
        n_oraciones += 1

        if len(documento_actual) == max_oracion_en_parrafo:
            documento.append(documento_actual)
            documento_actual = []

        if n_oraciones >= num_oraciones_por_dataset:
            break

    if documento_actual:
        documento.append(documento_actual)

    return documento#lista de (#num_oraciones_por_dataset) oraciones pegadas

# hacer instancias

In [ ]:
def dividir_en_splits(documentos, rng):
    """Mezcla los documentos y los reparte en train / test."""
    documentos = list(documentos)
    rng.shuffle(documentos)

    n_test = int(len(documentos) * proporcio_test)

    return {
        "test": documentos[:n_test],
        "train": documentos[n_test:],
    }


def hacer_instancias(documento, rng):#recortar_instancias # ?what

    if rng.random() < proba_loca:
        return [oracion.split() for oracion in documento]#lista de listas

    palabras = " ".join(documento).split()
    instancias = []
    inicio = 0
    while inicio < len(palabras):
        largo = rng.integers(min_palabra_instancia, max_palabra_instancia + 1)
        instancia = palabras[inicio:inicio + largo]
        # El último tramo puede quedar muy corto: lo descartamos
        if len(instancia) >= min_palabra_oracion:
            instancias.append(instancia)
        inicio += largo
    return instancias



def capitalizacion(palabra):
    #0: minúsculas, 1: primera mayúscula, 2: mixta (ej: iPhone), 3: todo mayúsculas.
    letras = [c for c in palabra if c.isalpha()]
    if not any(c.isupper() for c in letras):#ninguna mayonesa
        return 0
    if len(letras) > 1 and all(c.isupper() for c in letras):#todas mayo
        return 3
    if letras[0].isupper() and not any(c.isupper() for c in letras[1:]):#la primera sola
        return 1
    return 2#otra cosa

def etiquetar_palabra(palabra_cruda:str):

    punt_inicial = "¿" if palabra_cruda.startswith("¿") else ""
    punt_final = palabra_cruda[-1] if palabra_cruda[-1] in ".,?" else ""
    palabra_original = palabra_cruda.strip("¿?.,")

    return {
        "palabra": palabra_original.lower(),
        "palabra_original": palabra_original,
        "punt_inicial": punt_inicial,
        "punt_final": punt_final,
        "capitalizacion": capitalizacion(palabra_original),
    }

